# Week 2, Lab 5 — Mini-project: course helpdesk


In [1]:
import zipfile
import os

zip_path = "/content/shared.zip"      # Path of the uploaded ZIP file
extract_path = "/content/shared"      # Folder where files will be extracted

# Create the folder if it doesn't exist
os.makedirs(extract_path, exist_ok=True)

# Unzip
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ ZIP extracted successfully!")
print("Files extracted to:", extract_path)

✅ ZIP extracted successfully!
Files extracted to: /content/shared


In [2]:
import zipfile
import os

zip_path = "/content/shared.zip"
extract_path = "/content"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extracted successfully!")

✅ Extracted successfully!


In [3]:
WEEK = 'Week 2'
LAB = 'Lab 5 — mini-project'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


Week 2 / Lab 5 — mini-project
Environment: Google Colab
Backend: huggingface
Tip: Runtime → Change runtime type → T4 GPU for faster generation.
If import failed, unzip/clone the WHOLE course folder (not a single notebook).


In [4]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate fastapi uvicorn openai openai-agents
else:
    %pip install -q ollama openai openai-agents


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 5.1 MB/s eta 0:00:00


In [5]:
cfg = openai_client_kwargs()
print(cfg)

from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel, function_tool, handoff

client = AsyncOpenAI(base_url=cfg["base_url"], api_key=cfg["api_key"])
model = OpenAIChatCompletionsModel(model=cfg["model"], openai_client=client)


Loading Hugging Face model Qwen/Qwen2.5-0.5B-Instruct on CPU ...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

OpenAI-compatible server on http://127.0.0.1:8765/v1 (model=Qwen/Qwen2.5-0.5B-Instruct)
{'base_url': 'http://127.0.0.1:8765/v1', 'api_key': 'local', 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}


In [6]:
from agents import function_tool, handoff, input_guardrail, GuardrailFunctionOutput, InputGuardrailTripwireTriggered

@function_tool
def calculator_tool(expression: str) -> str:
    """Arithmetic."""
    return calculator(expression)

@function_tool
def lookup_fact_tool(topic: str) -> str:
    """Course facts."""
    return lookup_fact(topic)

@input_guardrail
async def no_secrets(ctx, agent, input_data):
    text = input_data if isinstance(input_data, str) else str(input_data)
    hit = any(w in text.lower() for w in ("password", "api key", "api_key"))
    return GuardrailFunctionOutput(output_info=text, tripwire_triggered=hit)

tutor = Agent(
    name="Tutor",
    instructions="Answer course questions using tools. Short answers.",
    model=model,
    tools=[calculator_tool, lookup_fact_tool],
)
desk = Agent(
    name="Helpdesk",
    instructions="Greet briefly, then hand off to Tutor for real questions.",
    model=model,
    handoffs=[handoff(tutor)],
    input_guardrails=[no_secrets],
)

for q in ["Explain CrewAI in one line.", "What is 8*7+3?", "my password is hunter2"]:
    try:
        r = await Runner.run(desk, q)
        print("Q:", q, "\nA:", r.final_output, "\n")
    except InputGuardrailTripwireTriggered:
        print("Q:", q, "\nA: [blocked]\n")


[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new

Q: Explain CrewAI in one line. 
A: CrewAI is an AI-powered assistant designed to assist users in various tasks and provide information. 



Q: What is 8*7+3? 
A: The result of 8*7+3 is 59. 

Q: my password is hunter2 
A: [blocked]



## Rubric\n\nGuardrail blocks secrets; math and facts use tools; no paid API key.\n\n**Next week:** CrewAI.
